# Lecture 02: LLM Architecture — Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_ML_Engineering/blob/main/lectures/01_a_ml_foundations/exercises/05_attention/notebook.ipynb)

In [ ]:
import os, importlib
if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_ML_Engineering/main/lectures/01_a_ml_foundations/exercises/05_attention/utils.py",
        "utils.py"
    )
    importlib.invalidate_caches()

import utils
importlib.reload(utils)
from utils import (
    test_causal_mask, test_attention_pattern,
    test_single_head_attention, test_multi_head_attention,
)

In [ ]:
import torch
import torch.nn as nn
import einops
import math
import matplotlib.pyplot as plt

## Attention: Moving Information Between Positions

Attention is the mechanism that lets tokens **look at other tokens**. It is the only part of a transformer that moves information between sequence positions. Everything else (MLP, LayerNorm) operates on each position independently.

The core idea:
1. Each token produces a **query** ("what am I looking for?") and a **key** ("what do I contain?")
2. Query-key dot products determine **attention scores**: how much each token attends to every other
3. Scores become a probability distribution (via softmax)
4. Each token computes a weighted average of **value** vectors (the actual information to move)

<img src="https://raw.githubusercontent.com/callummcdougall/computational-thread-art/master/example_images/misc/transformer-attn-new.png" width="900">

## Causal Masking

In autoregressive language models, token $t$ can only attend to tokens $\leq t$ (it can't look into the future). We enforce this by setting attention scores for future positions to $-\infty$ before the softmax, so they get probability 0.

$$\text{mask}_{ij} = \begin{cases} 0 & \text{if } i \geq j \\ -\infty & \text{if } i < j \end{cases}$$

## Exercise A: Causal Mask

Write a function that takes attention scores of shape `(batch, head, seq_q, seq_k)` and sets all entries where `seq_q < seq_k` to `-1e5` (our practical $-\infty$).

Use `torch.triu` to create the mask.

In [ ]:
def apply_causal_mask(attn_scores):
    """
    Args:
        attn_scores: (batch, head, seq_q, seq_k)
    Returns:
        masked scores with future positions set to -1e5
    """
    # TODO (~3 lines): create upper-triangular mask, apply it
    pass

In [ ]:
test_causal_mask(apply_causal_mask)

<details>
<summary><b>Hint</b></summary>

`torch.triu(ones_matrix, diagonal=1)` gives you a matrix that is 1 above the diagonal and 0 on/below. Use `.bool()` and `masked_fill_`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def apply_causal_mask(attn_scores):
    seq_len = attn_scores.size(-1)
    mask = torch.triu(torch.ones(seq_len, seq_len, device=attn_scores.device), diagonal=1).bool()
    attn_scores.masked_fill_(mask, -1e5)
    return attn_scores
```
</details>

---

## Exercise B: Attention Pattern

Given queries $Q$ and keys $K$ (both shape `(batch, seq, d_head)`), compute the causal attention pattern:

$$A = \text{softmax}\!\left(\frac{Q K^T}{\sqrt{d_{\text{head}}}} + \text{mask}\right)$$

The scaling by $\sqrt{d_{\text{head}}}$ prevents the dot products from growing too large, which would push softmax into saturation (near-zero gradients).

In [ ]:
def attention_pattern(Q, K, d_head):
    """
    Args:
        Q: (batch, seq, d_head) query vectors
        K: (batch, seq, d_head) key vectors
        d_head: dimension of each head (for scaling)
    Returns:
        (batch, seq_q, seq_k) attention weights (rows sum to 1)
    """
    # TODO (~3 lines): compute scaled dot-product attention with causal mask
    pass

In [ ]:
test_attention_pattern(attention_pattern)

<details>
<summary><b>Hint</b></summary>

Use `Q @ K.transpose(-2, -1)` for the dot products, scale, apply your `apply_causal_mask`, then softmax on `dim=-1`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def attention_pattern(Q, K, d_head):
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_head)
    scores = apply_causal_mask(scores)
    return scores.softmax(dim=-1)
```
</details>

In [ ]:
# Visualize a random attention pattern
torch.manual_seed(0)
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
pattern = attention_pattern(Q, K, d_head=16)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pattern[0].detach(), cmap='Blues')
ax.set(xlabel='Key (source)', ylabel='Query (destination)', title='Causal attention pattern')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

Notice the lower-triangular structure: each query can only attend to keys at the same or earlier position.

---

## Exercise C: Single-Head Attention

Now put it all together. Implement a single causal self-attention head as an `nn.Module`.

The full pipeline:
1. Project input $x$ into queries, keys, and values using weight matrices $W_Q, W_K, W_V$ (each `d_model -> d_head`)
2. Compute scaled dot-product attention with causal mask
3. Project the output back to `d_model` using $W_O$ (`d_head -> d_model`)

$$\text{head}(x) = \text{softmax}\!\left(\frac{(xW_Q)(xW_K)^T}{\sqrt{d_{\text{head}}}}\right) (xW_V) \, W_O$$

In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model=32, d_head=16):
        super().__init__()
        self.d_head = d_head
        self.W_Q = nn.Parameter(torch.randn(d_model, d_head) * 0.02)
        self.W_K = nn.Parameter(torch.randn(d_model, d_head) * 0.02)
        self.W_V = nn.Parameter(torch.randn(d_model, d_head) * 0.02)
        self.W_O = nn.Parameter(torch.randn(d_head, d_model) * 0.02)

    def forward(self, x):
        """
        Args:
            x: (batch, seq, d_model)
        Returns:
            (batch, seq, d_model)
        """
        # TODO: compute Q, K, V projections
        # TODO: compute attention pattern (scaled, masked, softmaxed)
        # TODO: apply attention to values and project output
        pass

In [ ]:
test_single_head_attention(SingleHeadAttention)

<details>
<summary><b>Hint</b></summary>

1. `Q = x @ self.W_Q` (and similarly for K, V)
2. Attention scores = `Q @ K.transpose(-2, -1) / sqrt(d_head)`, then mask, then softmax
3. Output = `(pattern @ V) @ self.W_O`
</details>

<details>
<summary><b>Solution</b></summary>

```python
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model=32, d_head=16):
        super().__init__()
        self.d_head = d_head
        self.W_Q = nn.Parameter(torch.randn(d_model, d_head) * 0.02)
        self.W_K = nn.Parameter(torch.randn(d_model, d_head) * 0.02)
        self.W_V = nn.Parameter(torch.randn(d_model, d_head) * 0.02)
        self.W_O = nn.Parameter(torch.randn(d_head, d_model) * 0.02)

    def forward(self, x):
        Q = x @ self.W_Q
        K = x @ self.W_K
        V = x @ self.W_V
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        mask = torch.triu(torch.ones(scores.size(-2), scores.size(-1), device=x.device), diagonal=1).bool()
        scores.masked_fill_(mask, -1e5)
        pattern = scores.softmax(dim=-1)
        z = pattern @ V
        return z @ self.W_O
```
</details>

---

## Exercise D: Multi-Head Attention

A single attention head can only compute one attention pattern. **Multi-head attention** runs `n_heads` heads in parallel, each with their own $W_Q, W_K, W_V, W_O$, and sums their outputs.

This lets the model attend to information from different representation subspaces at different positions simultaneously. One head might track syntactic relations, another semantic similarity, another positional proximity.

<img src="https://raw.githubusercontent.com/callummcdougall/computational-thread-art/master/example_images/misc/transformer-attn-21.png" width="1100">

The weight matrices now have an extra `head` dimension:
- `W_Q, W_K, W_V`: shape `(n_heads, d_model, d_head)`
- `W_O`: shape `(n_heads, d_head, d_model)`

Use `einops.einsum` for the batched matrix multiplies across heads. The named dimensions make the operations self-documenting:

```python
einops.einsum(x, W, "batch seq d_model, head d_model d_head -> batch seq head d_head")
```

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4, d_head=16):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_head
        self.W_Q = nn.Parameter(torch.randn(n_heads, d_model, d_head) * 0.02)
        self.W_K = nn.Parameter(torch.randn(n_heads, d_model, d_head) * 0.02)
        self.W_V = nn.Parameter(torch.randn(n_heads, d_model, d_head) * 0.02)
        self.W_O = nn.Parameter(torch.randn(n_heads, d_head, d_model) * 0.02)

    def forward(self, x):
        """
        Args:
            x: (batch, seq, d_model)
        Returns:
            (batch, seq, d_model)
        """
        # TODO: compute Q, K, V for all heads using einops.einsum
        # TODO: compute attention scores, scale, mask, softmax
        # TODO: apply attention to values
        # TODO: project output and sum over heads
        pass

In [ ]:
test_multi_head_attention(MultiHeadAttention)

<details>
<summary><b>Hint: step-by-step einsum patterns</b></summary>

```python
# Q, K, V projections
"batch seq d_model, head d_model d_head -> batch seq head d_head"

# Attention scores (Q @ K^T for each head)
"batch seq_q head d_head, batch seq_k head d_head -> batch head seq_q seq_k"

# Weighted sum of values
"batch head seq_q seq_k, batch seq_k head d_head -> batch seq_q head d_head"

# Output projection (sum over heads happens here)
"batch seq_q head d_head, head d_head d_model -> batch seq_q d_model"
```
</details>

<details>
<summary><b>Solution</b></summary>

```python
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4, d_head=16):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_head
        self.W_Q = nn.Parameter(torch.randn(n_heads, d_model, d_head) * 0.02)
        self.W_K = nn.Parameter(torch.randn(n_heads, d_model, d_head) * 0.02)
        self.W_V = nn.Parameter(torch.randn(n_heads, d_model, d_head) * 0.02)
        self.W_O = nn.Parameter(torch.randn(n_heads, d_head, d_model) * 0.02)

    def forward(self, x):
        Q = einops.einsum(x, self.W_Q, "batch seq d_model, head d_model d_head -> batch seq head d_head")
        K = einops.einsum(x, self.W_K, "batch seq d_model, head d_model d_head -> batch seq head d_head")
        V = einops.einsum(x, self.W_V, "batch seq d_model, head d_model d_head -> batch seq head d_head")

        scores = einops.einsum(Q, K, "batch seq_q head d_head, batch seq_k head d_head -> batch head seq_q seq_k")
        scores = scores / math.sqrt(self.d_head)
        mask = torch.triu(torch.ones(scores.size(-2), scores.size(-1), device=x.device), diagonal=1).bool()
        scores.masked_fill_(mask, -1e5)
        pattern = scores.softmax(dim=-1)

        z = einops.einsum(pattern, V, "batch head seq_q seq_k, batch seq_k head d_head -> batch seq_q head d_head")
        return einops.einsum(z, self.W_O, "batch seq_q head d_head, head d_head d_model -> batch seq_q d_model")
```
</details>

### Visualize multi-head attention patterns

Let's see what different heads attend to on random input.

In [ ]:
torch.manual_seed(42)
mha = MultiHeadAttention(d_model=64, n_heads=4, d_head=16)
x = torch.randn(1, 12, 64)

# Extract attention patterns by running forward manually
Q = einops.einsum(x, mha.W_Q, "batch seq d_model, head d_model d_head -> batch seq head d_head")
K = einops.einsum(x, mha.W_K, "batch seq d_model, head d_model d_head -> batch seq head d_head")
scores = einops.einsum(Q, K, "batch seq_q head d_head, batch seq_k head d_head -> batch head seq_q seq_k")
scores = scores / math.sqrt(mha.d_head)
mask = torch.triu(torch.ones(12, 12), diagonal=1).bool()
scores.masked_fill_(mask, -1e5)
patterns = scores.softmax(dim=-1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for h, ax in enumerate(axes):
    im = ax.imshow(patterns[0, h].detach(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {h}')
    ax.set(xlabel='Key', ylabel='Query')
plt.suptitle('Attention patterns across 4 heads (random weights)', y=1.02)
plt.tight_layout()
plt.show()